# Feature Selection for TCGA-BRCA Multimodal Data

This notebook explores feature selection methods for high-dimensional molecular data.
The goal is to identify the most informative features for PAM50 subtype classification,
and determine sensible input dimensionality for model encoders.

**Data used:** miRNA expression (1881 features), mRNA FPKM-UQ (~60k genes)

**Label:** PAM50 breast cancer molecular subtype (LumA, LumB, Her2, Basal, Normal)

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import kruskal

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

DATA_DIR = Path('/workspace/data/xenabrowser/TCGA-BRCA')

def sample_to_patient(s: str) -> str:
    parts = str(s).split('-')
    return '-'.join(parts[:3]) if len(parts) >= 3 else s

print('Ready.')

## 1. Load miRNA Data + PAM50 Labels

In [ ]:
# Load miRNA (features are rows, samples are columns)
mirna_raw = pd.read_csv(DATA_DIR / 'TCGA-BRCA.mirna.tsv', sep='\t', index_col=0)
# Transpose so rows = samples, cols = features
mirna = mirna_raw.T.copy()
mirna.index = mirna.index.map(sample_to_patient)
mirna.index.name = 'patient_id'
# Deduplicate (multiple sample types per patient — keep first)
mirna = mirna[~mirna.index.duplicated(keep='first')]
print(f'miRNA: {mirna.shape[0]} patients × {mirna.shape[1]} features')

# Load PAM50 labels
pam50 = pd.read_csv(DATA_DIR / 'pam50.tsv', sep='\t')
pam50['patient_id'] = pam50['sample'].apply(sample_to_patient)
pam50 = pam50.drop_duplicates('patient_id').set_index('patient_id')['PAM50']
print(f'PAM50: {len(pam50)} patients, classes: {sorted(pam50.unique())}')

# Align
common = mirna.index.intersection(pam50.index)
X_mirna = mirna.loc[common]
y = pam50.loc[common]
print(f'\nAligned: {len(common)} patients')
print(y.value_counts())

## 2. Baseline Data Quality

In [ ]:
# Missing values
nan_frac = X_mirna.isna().mean()
print(f'Features with any NaN: {(nan_frac > 0).sum()}')
print(f'Features with >10% NaN: {(nan_frac > 0.1).sum()}')

# Zero-inflated features
zero_frac = (X_mirna == 0).mean()
print(f'\nFeatures with >90% zeros: {(zero_frac > 0.9).sum()}')
print(f'Features with >50% zeros: {(zero_frac > 0.5).sum()}')

# Distribution of values
flat = X_mirna.values.flatten()
flat_nonzero = flat[flat > 0]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(flat_nonzero, bins=80, color='steelblue', alpha=0.8, edgecolor='white')
axes[0].set_xlabel('miRNA expression (raw)')
axes[0].set_title('Expression Distribution (non-zero values)')

axes[1].hist(np.log1p(flat_nonzero), bins=80, color='coral', alpha=0.8, edgecolor='white')
axes[1].set_xlabel('log1p(miRNA expression)')
axes[1].set_title('log1p-transformed Distribution')

for ax in axes:
    ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# Apply log1p and fill NaN with 0
X = np.log1p(X_mirna.fillna(0).values)
feature_names = X_mirna.columns.tolist()
print(f'X shape: {X.shape}')

## 3. Variance-Based Filtering

In [ ]:
variances = X.var(axis=0)
sorted_var = np.sort(variances)[::-1]
cumvar = np.cumsum(sorted_var) / sorted_var.sum()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(sorted_var, color='steelblue')
axes[0].set_xlabel('Feature rank (by variance)')
axes[0].set_ylabel('Variance')
axes[0].set_title('Variance Scree Plot')
axes[0].axvline(x=512, color='red', linestyle='--', label='top 512')
axes[0].axvline(x=256, color='orange', linestyle='--', label='top 256')
axes[0].legend()

axes[1].plot(cumvar, color='seagreen')
axes[1].axhline(y=0.95, color='red', linestyle='--', label='95% variance')
axes[1].axhline(y=0.99, color='orange', linestyle='--', label='99% variance')
n_95 = np.searchsorted(cumvar, 0.95) + 1
n_99 = np.searchsorted(cumvar, 0.99) + 1
axes[1].set_xlabel('Number of features')
axes[1].set_ylabel('Cumulative variance explained')
axes[1].set_title('Cumulative Variance')
axes[1].legend()

for ax in axes:
    ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

print(f'Features for 95% variance: {n_95}')
print(f'Features for 99% variance: {n_99}')
print(f'Top 512 features capture:  {cumvar[511]*100:.1f}% of variance')
print(f'Top 256 features capture:  {cumvar[255]*100:.1f}% of variance')

## 4. Differential Expression: Kruskal-Wallis Test

Find features that differ significantly across PAM50 subtypes.

In [ ]:
from scipy.stats import kruskal
from statsmodels.stats.multitest import multipletests

classes = sorted(y.unique())
X_df = pd.DataFrame(X, index=common, columns=feature_names)

pvals = []
for feat in feature_names:
    groups = [X_df.loc[y == c, feat].values for c in classes]
    try:
        stat, p = kruskal(*groups)
    except Exception:
        p = 1.0
    pvals.append(p)

pvals = np.array(pvals)
_, padj, _, _ = multipletests(pvals, method='fdr_bh')

n_sig_01 = (padj < 0.01).sum()
n_sig_05 = (padj < 0.05).sum()
print(f'Significant features (FDR < 0.01): {n_sig_01}')
print(f'Significant features (FDR < 0.05): {n_sig_05}')

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(-np.log10(padj + 1e-300), bins=60, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(-np.log10(0.05), color='red', linestyle='--', label='FDR 0.05')
ax.axvline(-np.log10(0.01), color='orange', linestyle='--', label='FDR 0.01')
ax.set_xlabel('-log10(adjusted p-value)')
ax.set_ylabel('Number of features')
ax.set_title('Kruskal-Wallis Test: miRNA Differential Expression across PAM50 Subtypes')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

## 5. Mutual Information with PAM50 Labels

In [ ]:
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_enc = le.fit_transform(y)

mi_scores = mutual_info_classif(X, y_enc, discrete_features=False, random_state=42)
mi_series = pd.Series(mi_scores, index=feature_names).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

top_n = 30
mi_series.head(top_n).plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_title(f'Top {top_n} miRNAs by Mutual Information')
axes[0].set_xlabel('')
axes[0].set_ylabel('Mutual Information Score')
axes[0].tick_params(axis='x', labelsize=7, rotation=60)
axes[0].spines[['top', 'right']].set_visible(False)

# Cumulative MI vs number of features
cummi = mi_series.values.cumsum() / mi_series.sum()
axes[1].plot(range(1, len(cummi) + 1), cummi, color='seagreen')
axes[1].axhline(0.90, color='red', linestyle='--', label='90% MI')
axes[1].axhline(0.95, color='orange', linestyle='--', label='95% MI')
n_90 = np.searchsorted(cummi, 0.90) + 1
n_95_mi = np.searchsorted(cummi, 0.95) + 1
axes[1].set_xlabel('Number of features (ranked by MI)')
axes[1].set_ylabel('Cumulative MI fraction')
axes[1].set_title('Cumulative Mutual Information')
axes[1].legend()
axes[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

print(f'Features for 90% MI: {n_90}')
print(f'Features for 95% MI: {n_95_mi}')

## 6. LASSO Feature Selection

In [ ]:
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Multi-class LASSO via one-vs-rest is complex; use L1-penalised logistic regression instead
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# Fast scan over C values
Cs = [0.001, 0.01, 0.1, 1.0]
n_nonzero = []
cv_scores = []

for C in Cs:
    lr = LogisticRegression(penalty='l1', C=C, solver='saga', max_iter=2000,
                            multi_class='ovr', random_state=42)
    scores = cross_val_score(lr, X_scaled, y_enc, cv=5, scoring='f1_macro')
    lr.fit(X_scaled, y_enc)
    nz = (np.abs(lr.coef_) > 1e-5).any(axis=0).sum()
    cv_scores.append(scores.mean())
    n_nonzero.append(nz)
    print(f'C={C:.3f}: {nz:4d} features, F1-macro = {scores.mean():.3f} ± {scores.std():.3f}')

fig, ax1 = plt.subplots(figsize=(8, 4))
ax2 = ax1.twinx()
ax1.plot(Cs, n_nonzero, 'bo-', label='# features')
ax2.plot(Cs, cv_scores, 'rs--', label='F1-macro (5-fold CV)')
ax1.set_xscale('log')
ax1.set_xlabel('C (inverse regularisation strength)')
ax1.set_ylabel('Non-zero features', color='b')
ax2.set_ylabel('F1-macro', color='r')
ax1.set_title('L1 Logistic Regression: Feature Count vs. Performance')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right')
plt.tight_layout()
plt.show()

## 7. Random Forest Feature Importance

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Prefilter to top-500 by variance to keep RF tractable
top_var_idx = np.argsort(variances)[::-1][:500]
X_var = X[:, top_var_idx]
var_feature_names = [feature_names[i] for i in top_var_idx]

rf = RandomForestClassifier(n_estimators=200, max_depth=None, random_state=42, n_jobs=-1)
rf.fit(X_var, y_enc)

rf_importance = pd.Series(rf.feature_importances_, index=var_feature_names).sort_values(ascending=False)

top_n = 30
fig, ax = plt.subplots(figsize=(11, 5))
rf_importance.head(top_n).plot(kind='bar', ax=ax, color='seagreen', edgecolor='white', alpha=0.85)
ax.set_title(f'Top {top_n} miRNAs by Random Forest Importance (from top-500 by variance)')
ax.set_ylabel('Feature Importance')
ax.tick_params(axis='x', labelsize=7, rotation=60)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

# 5-fold CV on top-K by RF importance
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import GradientBoostingClassifier

print('Cross-validation F1-macro with top-K RF features:')
for k in [50, 100, 200, 500]:
    top_k = rf_importance.head(k).index
    X_k = X_var[:, [var_feature_names.index(f) for f in top_k]]
    clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    scores = cross_val_score(clf, X_k, y_enc, cv=5, scoring='f1_macro')
    print(f'  Top {k:3d}: F1-macro = {scores.mean():.3f} ± {scores.std():.3f}')

## 8. PCA Visualisation

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=50, random_state=42)
X_pca = pca.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

palette = sns.color_palette('tab10', len(classes))
for i, cls in enumerate(classes):
    mask = y.values == cls
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    label=cls, alpha=0.7, s=25, color=palette[i])
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
axes[0].set_title('PCA: miRNA (all features)')
axes[0].legend(fontsize=9)
axes[0].spines[['top', 'right']].set_visible(False)

# Scree
axes[1].bar(range(1, 21), pca.explained_variance_ratio_[:20] * 100,
            color='steelblue', edgecolor='white', alpha=0.85)
axes[1].set_xlabel('Principal Component')
axes[1].set_ylabel('Explained Variance (%)')
axes[1].set_title('PCA Scree (first 20 PCs)')
axes[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

print(f'Variance explained by top-10 PCs: {pca.explained_variance_ratio_[:10].sum()*100:.1f}%')

## 9. UMAP Visualisation

In [ ]:
try:
    import umap
    has_umap = True
except ImportError:
    has_umap = False
    print('umap-learn not installed; install with: pip install umap-learn')

if has_umap:
    reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.3)
    X_umap = reducer.fit_transform(X_pca)  # UMAP on top-50 PCs for speed

    fig, ax = plt.subplots(figsize=(8, 7))
    for i, cls in enumerate(classes):
        mask = y.values == cls
        ax.scatter(X_umap[mask, 0], X_umap[mask, 1],
                   label=cls, alpha=0.75, s=30, color=palette[i])
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    ax.set_title('UMAP: miRNA expression coloured by PAM50 subtype')
    ax.legend(fontsize=10)
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.show()

## 10. Feature Selection Method Comparison

In [ ]:
# Rank features by each method
rank_variance = pd.Series(
    {feature_names[i]: r for r, i in enumerate(np.argsort(variances)[::-1])},
    name='variance_rank',
)
rank_mi = pd.Series(
    {f: r for r, f in enumerate(mi_series.index)},
    name='mi_rank',
)
rank_kw = pd.Series(
    {feature_names[i]: r for r, i in enumerate(np.argsort(padj))},
    name='kw_rank',
)

rank_df = pd.concat([rank_variance, rank_mi, rank_kw], axis=1)

# Correlation between rankings
from scipy.stats import spearmanr
print('Spearman rank correlation between selection methods:')
methods = ['variance_rank', 'mi_rank', 'kw_rank']
for i, m1 in enumerate(methods):
    for m2 in methods[i+1:]:
        rho, pv = spearmanr(rank_df[m1], rank_df[m2])
        print(f'  {m1} vs {m2}: rho={rho:.3f}, p={pv:.2e}')

# Consensus: top-100 by each method
top100_var = set(np.argsort(variances)[::-1][:100])
top100_mi  = set([feature_names.index(f) for f in mi_series.head(100).index])
top100_kw  = set(np.argsort(padj)[:100])

consensus_100 = top100_var & top100_mi & top100_kw
consensus_any = top100_var | top100_mi | top100_kw

print(f'\nConsensus top-100 features (all 3 methods agree): {len(consensus_100)}')
print(f'Union of top-100 features (any method):           {len(consensus_any)}')

In [ ]:
# CV performance comparison across feature selection strategies
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

strategies = {
    'All features (1881)': list(range(len(feature_names))),
    'Top-512 (variance)':  list(np.argsort(variances)[::-1][:512]),
    'Top-256 (variance)':  list(np.argsort(variances)[::-1][:256]),
    'Top-100 (MI)':        [feature_names.index(f) for f in mi_series.head(100).index],
    'Top-100 (KW)':        list(np.argsort(padj)[:100]),
    'Consensus-top100':    list(consensus_100),
}

results = []
for name, idxs in strategies.items():
    X_sub = X[:, idxs]
    clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    scores = cross_val_score(clf, X_sub, y_enc, cv=5, scoring='f1_macro')
    results.append({'Strategy': name, 'n_features': len(idxs),
                    'F1_macro': scores.mean(), 'F1_std': scores.std()})
    print(f'{name:30s}: n={len(idxs):4d}, F1={scores.mean():.3f} ± {scores.std():.3f}')

results_df = pd.DataFrame(results)

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(results_df['Strategy'], results_df['F1_macro'],
              yerr=results_df['F1_std'], capsize=5,
              color='steelblue', edgecolor='white', alpha=0.85)
ax.set_ylabel('F1-macro (5-fold CV)')
ax.set_title('RF Performance by Feature Selection Strategy')
ax.set_xticklabels(results_df['Strategy'], rotation=25, ha='right', fontsize=9)
ax.set_ylim(0, 1)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

## 11. Top Features: Expression Patterns by Subtype

In [ ]:
top_feats = mi_series.head(20).index.tolist()
X_top = X_df[top_feats].copy()
X_top['PAM50'] = y.values

X_melt = X_top.melt(id_vars='PAM50', var_name='miRNA', value_name='log1p_expression')

fig, ax = plt.subplots(figsize=(14, 5))
sns.boxplot(
    data=X_melt, x='miRNA', y='log1p_expression', hue='PAM50',
    ax=ax, palette='tab10', linewidth=0.8, fliersize=2,
)
ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha='right', fontsize=8)
ax.set_title('Top-20 MI miRNAs: Expression by PAM50 Subtype')
ax.set_ylabel('log1p(expression)')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

## 12. Summary and Recommendations

| Method | Recommended N | Notes |
|---|---|---|
| Variance filtering | **512** | Simple, fast; retains ~X% of total variance |
| Mutual information | **100–256** | Most label-relevant; computationally heavier |
| KW differential | **~sig@FDR<0.01** | Statistically motivated |
| Consensus (all 3) | **~50–100** | Highest confidence; may over-prune |

### Recommendations for encoder input
1. **Use `max_seq_len=512`** (already in config) — good balance of variance coverage and SCBert memory.
2. **Pre-filter by variance** (top-512 by std across training patients) in the parser before feeding to the encoder — removes uninformative zero-heavy miRNAs.
3. **Optionally pre-select by MI** on the training fold during k-fold preprocessing — use the `data/preprocessing.py` pipeline to compute MI per fold and save selected feature indices.
4. **Do NOT select features on the full dataset** — always compute selection statistics on training data only to avoid label leakage.

### Next steps
- Repeat this analysis for mRNA (FPKM) data — ~60k genes → likely need stricter filtering
- Extend to protein and methylation when adding those modalities
- Save selected feature lists to `data/configs/BRCA/selected_features/` and load them in the parser